In [ ]:
%pip install transformers[torch] scikit-learn "accelerate>=1.1.0" --quiet

In [ ]:
import os
import random
import logging
from collections import Counter
from urllib import request

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

logging.basicConfig(level=logging.WARNING)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [ ]:
CONFIG = {
    "model_name": "FacebookAI/roberta-base",
    "max_length": 128,
    "num_epochs": 3,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "seed": 67,
    "test_size": 0.20,
    "thresh_start": 0.05,
    "thresh_end": 0.95,
    "thresh_step": 0.01,
    "output_dir": "./data",
}

seed = CONFIG["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

os.makedirs(CONFIG["output_dir"], exist_ok=True)

In [ ]:
tsv_path = "./dontpatronizeme_pcl.tsv"
if not os.path.exists(tsv_path):
    print("downloading dontpatronizeme_pcl.tsv")
    tsv_url = "https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv"
    with request.urlopen(tsv_url) as f, open(tsv_path, "wb") as out:
        out.write(f.read())

tsv_df = pd.read_csv(
    tsv_path,
    sep="\t",
    header=None,
    names=["par_id", "art_id", "keyword", "country", "text", "orig_label"],
    skiprows=4,
    dtype={"par_id": str},
)

tsv_df["label"] = tsv_df["orig_label"].apply(lambda x: 0 if int(x) <= 1 else 1)
print(f"TSV loaded: {len(tsv_df)} rows")
print("label dist:", Counter(tsv_df["label"]))

TSV loaded: 10469 rows
label dist: Counter({0: 9476, 1: 993})


In [ ]:
split_base = "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/"

def download_split_csv(fname):
    if not os.path.exists(f"./{fname}"):
        print(f"downloading {fname}")
        with request.urlopen(split_base + fname) as f, open(f"./{fname}", "wb") as out:
            out.write(f.read())

download_split_csv("train_semeval_parids-labels.csv")
download_split_csv("dev_semeval_parids-labels.csv")

def split_to_df(split_csv, tsv_df):
    split_ids = pd.read_csv(split_csv, dtype={"par_id": str})[["par_id"]]
    merged = split_ids.merge(tsv_df[["par_id", "text", "label"]], on="par_id", how="left")
    merged["text"] = merged["text"].fillna("").astype(str)
    return merged.reset_index(drop=True)

train_ids = pd.read_csv("./train_semeval_parids-labels.csv", dtype={"par_id": str})
dev_ids   = pd.read_csv("./dev_semeval_parids-labels.csv",   dtype={"par_id": str})

traindf = split_to_df("./train_semeval_parids-labels.csv", tsv_df)
devdf   = split_to_df("./dev_semeval_parids-labels.csv",   tsv_df)

assert len(traindf) == len(train_ids), f"train count mismatch: {len(traindf)}, {len(train_ids)}"
assert len(devdf)   == len(dev_ids),   f"dev count mismatch: {len(devdf)}, {len(dev_ids)}"
assert traindf["par_id"].tolist() == train_ids["par_id"].tolist(), "train par_id order mismatch"
assert devdf["par_id"].tolist()   == dev_ids["par_id"].tolist(),   "dev par_id order mismatch"

print(f"train: {len(traindf)}  dev: {len(devdf)}")
print("train labels:", Counter(traindf["label"]))
print("dev labels:", Counter(devdf["label"]))

train: 8375  dev: 2094
train labels: Counter({0: 7581, 1: 794})
dev labels: Counter({0: 1895, 1: 199})


In [ ]:
if not os.path.exists("./task4_test.tsv"):
    print("downloading task4_test.tsv")
    with request.urlopen("https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/TEST/task4_test.tsv") as f, open("./task4_test.tsv", "wb") as out:
        out.write(f.read())

test_df = pd.read_csv(
    "./task4_test.tsv",
    sep="\t",
    header=None,
    names=["par_id", "art_id", "keyword", "country", "text"],
    dtype={"par_id": str},
).reset_index(drop=True)

print(f"test: {len(test_df)} rows")

test: 3832 rows


In [ ]:
def sanitize_text_series(series):
    return series.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

n_train, n_dev, n_test = len(traindf), len(devdf), len(test_df)

for df in [traindf, devdf, test_df]:
    df["text"] = sanitize_text_series(df["text"])

assert len(traindf) == n_train
assert len(devdf)   == n_dev
assert len(test_df) == n_test

In [ ]:
train_df, val_df = train_test_split(
    traindf,
    test_size=CONFIG["test_size"],
    stratify=traindf["label"],
    random_state=CONFIG["seed"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"train: {len(train_df)}, val: {len(val_df)}, test: {len(devdf)}")
print(f"training positive: {sum(train_df.label == 1)}, negative: {sum(train_df.label == 0)}")

train: 6700, val: 1675, test: 2094
training positive: 635, negative: 6065


In [ ]:
cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df["label"].values)
class_weights = torch.tensor(cw, dtype=torch.float)

print(f"class weights: negative={cw[0]:.3f}, positive={cw[1]:.3f}")

class weights: negative=0.552, positive=5.276


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

In [ ]:
class PCL(Dataset):
    def __init__(self, raw_texts, targets):
        super().__init__()
        self.targets = targets
        self.encoded = tokenizer(
            raw_texts,
            max_length=CONFIG["max_length"],
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

    def __len__(self) -> int:
        return self.encoded["input_ids"].shape[0]

    def __getitem__(self, index: int) -> dict:
        batch_entry = {key: tensor[index] for key, tensor in self.encoded.items()}

        if self.targets is not None:
            batch_entry["labels"] = torch.tensor(self.targets[index], dtype=torch.long)

        return batch_entry


class ImbalanceTrainer(Trainer):
    def __init__(self, *args, class_weights: torch.Tensor, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs: bool = False, num_items_in_batch=None):
        ground_labels = inputs.pop("labels")
        model_out = model(**inputs)
        weights = self._class_weights.to(model_out.logits.device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        loss_val = criterion(model_out.logits, ground_labels)
        return (loss_val, model_out) if return_outputs else loss_val



In [ ]:
train_dataset = PCL(train_df["text"].tolist(), train_df["label"].tolist())
val_dataset   = PCL(val_df["text"].tolist(),   val_df["label"].tolist())
dev_dataset   = PCL(devdf["text"].tolist(),     devdf["label"].tolist())
test_dataset  = PCL(test_df["text"].tolist(), targets=None)

print(f"train={len(train_dataset)}  val={len(val_dataset)}  dev={len(dev_dataset)}  test={len(test_dataset)}")

train=6700  val=1675  dev=2094  test=3832


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"], num_labels=2
)

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"] * 2,
    learning_rate=CONFIG["learning_rate"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    seed=CONFIG["seed"],
    fp16=torch.cuda.is_available(),
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
trainer = ImbalanceTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    class_weights=class_weights,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.649707,0.446472
2,0.523262,0.880621
3,0.304334,0.998785


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1257, training_loss=0.46146125933814824, metrics={'train_runtime': 263.2922, 'train_samples_per_second': 76.341, 'train_steps_per_second': 4.774, 'total_flos': 1322133053184000.0, 'train_loss': 0.46146125933814824, 'epoch': 3.0})

In [ ]:
def get_probs(trainer, dataset):
    output = trainer.predict(dataset)
    logits = torch.tensor(output.predictions)
    probs = F.softmax(logits, dim=-1).numpy()
    return probs[:, 1], output.label_ids

def pcl_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average=None, labels=[0, 1], zero_division=0)[1]

val_probs, val_true = get_probs(trainer, val_dataset)

val_preds_05 = (val_probs >= 0.5).astype(int)
print(f"val threshold 0.5:  F1 PCL: {pcl_f1(val_true, val_preds_05):.5f}")

best_threshold, best_f1 = 0.5, 0.0

for t in np.arange(CONFIG["thresh_start"], CONFIG["thresh_end"], CONFIG["thresh_step"]):
    preds = (val_probs >= t).astype(int)
    f = pcl_f1(val_true, preds)
    if f > best_f1:
        best_f1 = f
        best_threshold = round(float(t), 5)

print(f"best threshold: {best_threshold}  F1: {best_f1:.5f}")

val_preds_tuned = (val_probs >= best_threshold).astype(int)
print(f"val threshold: {best_threshold}  F1 PCL: {pcl_f1(val_true, val_preds_tuned):.5f}")

val threshold 0.5:  F1 PCL: 0.45149
best threshold: 0.68  F1: 0.49649
val threshold: 0.68  F1 PCL: 0.49649


In [ ]:
dev_probs, dev_true = get_probs(trainer, dev_dataset)
dev_preds = (dev_probs >= best_threshold).astype(int)

print(f"dev F1 PCL: {pcl_f1(dev_true, dev_preds)}")

assert len(dev_preds) == 2094, f"dev length check failed: expected 2094 but got {len(dev_preds)}"

dev_txt_path = os.path.join(CONFIG["output_dir"], "dev.txt")
with open(dev_txt_path, "w") as fh:
    fh.writelines(str(pred) + "\n" for pred in dev_preds)

print(f"saved dev.txt ({len(dev_preds)} lines)")

dev F1 PCL: 0.4808743169398907
saved dev.txt (2094 lines)


In [ ]:
test_probs, _ = get_probs(trainer, test_dataset)
test_preds = (test_probs >= best_threshold).astype(int)

assert len(test_preds) == 3832, f"test length check failed: expected 3832 but got {len(test_preds)}"

test_txt_path = os.path.join(CONFIG["output_dir"], "test.txt")
with open(test_txt_path, "w") as f:
    f.writelines(str(p) + "\n" for p in test_preds)

print(f"saved test.txt ({len(test_preds)} lines)")

saved test.txt (3832 lines)
